# Task 2A - 04 Validation design and baselines
## Validation design
The real task: train on everything up to ISO week 13, forecast the next **10 weeks**. We reproduce that **forward in time**:
- 34 rolling origins (Mondays, every 2 weeks, 2024-10-07 ... 2026-01-12); at each, train only on days *before* the cut, forecast the following 10 ISO weeks;
- score weekly depot x brand totals (680 weekly forecasts per brand) with **MAE, RMSE, WAPE and bias** so conclusions do not depend on one metric (the official metric is not stated);
- origins before spring 2025 can only learn from one festival season, so results there are pessimistic - the final model has two seasons of each festival.
No random folds are used: they would leak neighbouring days of the same festival run-up.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2a')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib, json
from task2a_common import *
from task2a_features import *
from task2a_models import *
from task2a_validation import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (INTERIM, METRICS, FIGURES): d.mkdir(parents=True, exist_ok=True)
o, cal = load_orders(); cal2 = enrich_calendar(cal); panel = build_panel(o, cal2)


In [2]:
print(len(ORIGINS), 'origins:', ORIGINS[0].date(), '->', ORIGINS[-1].date(), '| horizon', HORIZON_WEEKS, 'weeks')
print('last window ends', (ORIGINS[-1] + pd.Timedelta(weeks=HORIZON_WEEKS) - pd.Timedelta(days=1)).date(), '(data ends', panel.date.max().date(), ')')

34 origins: 2024-10-07 -> 2026-01-12 | horizon 10 weeks
last window ends 2026-03-22 (data ends 2026-03-28 )


## Baselines (what a planner would do without a model)

In [3]:
res = []
for b in BRANDS:
    res.append(backtest(panel, b, lambda: NaiveRecent(8), name='naive: last 8 weeks mean'))
    res.append(backtest(panel, b, lambda: NaiveRecent(26), name='naive: last 26 weeks mean'))
    res.append(backtest(panel, b, lambda: SeasonalNaive(), name='seasonal naive (same ISO week last year x growth)'))
bt0 = pd.concat(res); bt0.to_pickle(METRICS / 'task2a_baselines_weeks.pkl')
s0 = summarize(bt0); s0.to_csv(METRICS / 'task2a_baselines.csv', index=False)
s0.pivot(index='model', columns='brand', values='WAPE').round(4)

brand,Fresh,Style,Tech
model,,,
naive: last 26 weeks mean,0.0601,0.1058,0.3031
naive: last 8 weeks mean,0.0631,0.1132,0.3162
seasonal naive (same ISO week last year x growth),0.0809,0.0936,0.3418


In [4]:
s0.sort_values(['brand', 'WAPE'])

,model,brand,n,MAE,RMSE,WAPE,bias
0,naive: last 26 weeks mean,Fresh,680.0,44.6456,68.4452,0.0601,-0.0191
3,naive: last 8 weeks mean,Fresh,680.0,46.8596,70.2724,0.0631,-0.0114
6,seasonal naive (same ISO week last year x growth),Fresh,680.0,60.0515,79.7028,0.0809,-0.0505
7,seasonal naive (same ISO week last year x growth),Style,680.0,10.4683,15.7458,0.0936,-0.0160
1,naive: last 26 weeks mean,Style,680.0,11.8275,18.6664,0.1058,-0.0100
4,naive: last 8 weeks mean,Style,680.0,12.6653,19.4901,0.1132,-0.0090
2,naive: last 26 weeks mean,Tech,680.0,7.9024,9.5350,0.3031,-0.0391
5,naive: last 8 weeks mean,Tech,680.0,8.2439,9.9810,0.3162,-0.0170
8,seasonal naive (same ISO week last year x growth),Tech,680.0,8.9110,11.0984,0.3418,-0.0475


In [5]:
festival_split(bt0).pivot_table(index=['brand', 'model'], columns='bucket', values='WAPE').round(3)

bucket                                                   festival run-up weeks  other weeks
brand model                                                                                
Fresh naive: last 26 weeks mean                                          0.140        0.047
      naive: last 8 weeks mean                                           0.135        0.051
      seasonal naive (same ISO week last year x growth)                  0.155        0.069
Style naive: last 26 weeks mean                                          0.201        0.089
      naive: last 8 weeks mean                                           0.207        0.097
      seasonal naive (same ISO week last year x growth)                  0.140        0.086
Tech  naive: last 26 weeks mean                                          0.297        0.304
      naive: last 8 weeks mean                                           0.321        0.316
      seasonal naive (same ISO week last year x growth)                  0.415        0.330

**Reading.** Constant-level baselines are acceptable in calm weeks but fail in festival run-up weeks (Fresh error about 3x larger than in other weeks). Seasonal-naive is the best baseline for Style (festival weeks repeat) but the worst for Fresh and Tech, because it copies last year's week even when the festival or the number of operating days has moved. These are the numbers the model has to beat (notebook 05).